In [1]:
%%time
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import date
import time

# ----- S&P 500 -----
sp500_df = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")[0]
tickers = sp500_df['Symbol'].str.replace('.', '-', regex=False).tolist()

start_date = "2024-01-01"
end_date   = "2025-07-19" #date.today().isoformat()

all_data = pd.DataFrame()
failed_tickers = []

for ticker in tickers:
    try:
        print(f"📥 Downloading {ticker}")
        data = yf.download(ticker, start=start_date, end=end_date, progress=False, auto_adjust=True)
        if not data.empty:
            all_data[ticker] = data['Close']
        else:
            print(f"⚠️ No data for {ticker}")
            failed_tickers.append(ticker)
    except Exception as e:
        print(f"❌ Failed {ticker}: {e}")
        failed_tickers.append(ticker)
    time.sleep(1)

all_data.index.name = "Date"


valid_cols = all_data.notna().mean() >= 0.5
filtered_data = all_data.loc[:, valid_cols]

print(f"\n✅ obtained brands: {filtered_data.shape[1]} / {len(tickers)}")
print(f"🧹 excluded brands: {len(tickers) - filtered_data.shape[1]}")
print(f"🚫 failed brands: {len(failed_tickers)}")

# ----- 補間 -----
filled_data = filtered_data.ffill().bfill()

# ----- numpy形式に整形 -----
X = filled_data.to_numpy()
print(f"✅ data shape: {X.shape}")

# ----- 保存 -----
filled_data.to_csv("sp500.csv", encoding="utf-8-sig")
print("💾 CSV saved: sp500.csv")


HTTPError: HTTP Error 403: Forbidden